<a href="https://colab.research.google.com/github/divyankapawaskar/splink-workshop-binder/blob/main/Splink_synthetic_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Step 1: Import necessary python packages

!pip install splink
from IPython.display import display

import pandas as pd
import splink
import numpy as np


In [4]:

#Step 2: Import datasets to be linked ----------------------------------------------

# Load crash dataset

df_crash = pd.read_csv("/content/synthetic_crash.csv", index_col = 0)

# Load hospital records dataset

df_hosp = pd.read_csv("/content/synthetic_hosp.csv", index_col = 0)

# Check dimensions of datasets (number of rows and columns)

print(f"\nCrash records: {df_crash.shape[0]} rows x {df_crash.shape[1]} columns")
print(f"Hospital records: {df_hosp.shape[0]} rows x {df_hosp.shape[1]} columns")



Crash records: 1000 rows x 11 columns
Hospital records: 1000 rows x 11 columns


In [42]:
from IPython.display import display #Enables us to view 2 datasets in the output

# View crash and hospital data

display(df_crash.head(5))
display(df_hosp.head(5))

,firstname,lastname,sex,age,dob,role,admit_date,admit_hr,county,state,zip,unique_id,source
record_id,,,,,,,,,,,,,
1,<NA>,<NA>,M,19,2005-03-24,Passenger,2024-02-22,21,westchester,NY,<NA>,crash_1,crash
2,Edward,Thompson,F,55,1969-09-04,Driver,2024-07-12,2,queens,NY,14604,crash_2,crash
702,Edward,Lewis,M,68,1956-01-27,Driver,2024-10-05,1,richmond,NY,<NA>,crash_702,crash
712,Barbara,Nguyen,M,31,1993-06-03,Driver,2024-09-28,2,onondaga,NY,<NA>,crash_712,crash
3,Sandra,Harris,M,61,1963-04-22,Driver,2024-06-15,1,kings,NY,11201,crash_3,crash


,firstname,lastname,sex,age,dob,role,admit_date,admit_hr,county,state,zip,unique_id,source
record_id,,,,,,,,,,,,,
1,Kenneth,Davis,M,19,2005-03-24,Passenger,2024-02-24,0,westchester,NY,<NA>,hosp_1,hosp
2,Edward,Thompson,F,55,1969-09-04,Driver,2024-07-14,5,queens,NY,14604,hosp_2,hosp
3,Sandra,Harris,M,61,1963-04-22,Driver,2024-06-16,23,kings,NY,11201,hosp_3,hosp
4,Timotiy,Brown,F,18,2006-11-14,Driver,2024-11-02,14,new york,NY,10002,hosp_4,hosp
5,Steven,Hill,M,70,1954-09-25,Driver,2024-09-29,21,kings,NY,10002,hosp_5,hosp


In [44]:
# Step 3: Rename columns so both datasets share the same column names for linkage

#Crash Columns:

df_crash.rename(columns={
    'record_id':    'unique_id',
    'first_name':   'firstname',
    'last_name':    'lastname',
    'gender':       'sex',

    'bday':         'dob',
    'crash_date':   'admit_date',
    'crash_hour':   'admit_hr',


}, inplace=True)

# Hospital Columns:

df_hosp.rename(columns={
    'patient_record_id': 'unique_id',
    'first_name':        'firstname',
    'last_name':         'lastname',
    'gender':            'sex',

    'bday':              'dob',
    'crash_date':      'admit_date',
    'crash_hour':      'admit_hr',


}, inplace=True)




In [45]:
# Check that column names match after renaming

from IPython.display import display
display(df_crash.head(2))
display(df_hosp.head(2))

,firstname,lastname,sex,age,dob,role,admit_date,admit_hr,county,state,zip,unique_id,source
record_id,,,,,,,,,,,,,
1,<NA>,<NA>,M,19,2005-03-24,Passenger,2024-02-22,21,westchester,NY,<NA>,crash_1,crash
2,Edward,Thompson,F,55,1969-09-04,Driver,2024-07-12,2,queens,NY,14604,crash_2,crash


,firstname,lastname,sex,age,dob,role,admit_date,admit_hr,county,state,zip,unique_id,source
record_id,,,,,,,,,,,,,
1,Kenneth,Davis,M,19,2005-03-24,Passenger,2024-02-24,0,westchester,NY,<NA>,hosp_1,hosp
2,Edward,Thompson,F,55,1969-09-04,Driver,2024-07-14,5,queens,NY,14604,hosp_2,hosp


In [46]:
# Step 4: Check and convert data types for each column
# Best practice is to convert everything to strings or numeric types

print(df_crash.dtypes) #this code prints the column types
print(df_hosp.dtypes)

firstname     string[python]
lastname      string[python]
sex           string[python]
age                    Int64
dob                   object
role          string[python]
admit_date            object
admit_hr               Int64
county        string[python]
state         string[python]
zip                    Int64
unique_id             object
source                object
dtype: object
firstname     string[python]
lastname      string[python]
sex           string[python]
age                    Int64
dob                   object
role          string[python]
admit_date            object
admit_hr               Int64
county        string[python]
state         string[python]
zip                    Int64
unique_id             object
source                object
dtype: object


In [55]:
# Step 5: Convert columns to correct types

# Explanation for these choices:
# - Text columns are converted to `string` to ensure consistent handling and optimize text operations.
# - Numeric columns are converted to `Int64` (nullable integer) for efficient mathematical operations and to properly represent missing values (NaN).
# - Dates are temporarily converted to datetime objects for standardization, then back to formatted strings (%Y-%m-%d) to ensure consistency across datasets and compatibility with Splink's custom comparison functions.

string_cols = ['firstname', 'lastname', 'sex', 'dob', 'admit_date', 'county', 'role', 'state']
int_cols    = ['age', 'admit_hr', 'zip']

for col in string_cols:
    if col in df_crash.columns:
        df_crash[col] = df_crash[col].astype('string')
    if col in df_hosp.columns:
        df_hosp[col] = df_hosp[col].astype('string')

for col in int_cols:
    if col in df_crash.columns:
        df_crash[col] = pd.to_numeric(df_crash[col], errors='coerce').astype('Int64')
    if col in df_hosp.columns:
        df_hosp[col] = pd.to_numeric(df_hosp[col], errors='coerce').astype('Int64')

print(df_crash.dtypes)
print(df_hosp.dtypes)



firstname     string[python]
lastname      string[python]
sex           string[python]
age                    Int64
dob           string[python]
role          string[python]
admit_date    string[python]
admit_hr               Int64
county        string[python]
state         string[python]
zip                    Int64
unique_id             object
source                object
dtype: object
firstname     string[python]
lastname      string[python]
sex           string[python]
age                    Int64
dob           string[python]
role          string[python]
admit_date    string[python]
admit_hr               Int64
county        string[python]
state         string[python]
zip                    Int64
unique_id             object
source                object
dtype: object


In [58]:
# Step 6
#Match crash and hosp sex

df_hosp['sex'] = df_hosp['sex'].astype('string').replace({'Male': 'M', 'Female': 'F'})

#lowercase 'county'

for _df in (df_crash, df_hosp):
    _df['county'] = _df['county'].str.lower()

#Standardize date formats
df_crash['admit_date'] = pd.to_datetime(df_crash['admit_date'], errors='coerce').dt.strftime('%Y-%m-%d').astype('string')
df_hosp['admit_date']  = pd.to_datetime(df_hosp['admit_date'],  errors='coerce').dt.strftime('%Y-%m-%d').astype('string')

df_crash['dob'] = pd.to_datetime(df_crash['dob'], errors='coerce').dt.strftime('%Y-%m-%d').astype('string')
df_hosp['dob']  = pd.to_datetime(df_hosp['dob'],  errors='coerce').dt.strftime('%Y-%m-%d').astype('string')

# %%
# Step 7: Add source markers and splink IDs
df_crash['unique_id'] = 'crash_' + df_crash.index.astype(str)
df_hosp['unique_id']  = 'hosp_'  + df_hosp.index.astype(str)

df_crash['source'] = 'crash'
df_hosp['source']  = 'hosp'

# Verify that all columns are now numeric or string, and match between datasets
df_crash.dtypes
df_hosp.dtypes

,0
firstname,string[python]
lastname,string[python]
sex,string[python]
age,Int64
dob,string[python]
role,string[python]
admit_date,string[python]
admit_hr,Int64
county,string[python]
state,string[python]


In [59]:
# Step 8: Import splink modules
from splink import DuckDBAPI
from splink.exploratory import completeness_chart
from splink.exploratory import profile_columns

# Step 9: Generate and inspect charts to understand your data -----------------------

# Completeness Chart -- check data missingness and completeness
completeness_chart(
    [df_crash, df_hosp],
    db_api = DuckDBAPI(),
    table_names_for_chart = ["crash", "hosp"]).save("completeness_chart.html")

# Profile Columns Charts -- shows distributions of values for each variable
profile_columns(df_crash, db_api = DuckDBAPI()).save("profile_columns_crash.html")
profile_columns(df_hosp, db_api = DuckDBAPI()).save("profile_columns_hcup.html")

# Import Splink modules
from splink import block_on
from splink.blocking_analysis import cumulative_comparisons_to_be_scored_from_blocking_rules_chart
import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on

# Choose API
db_api = DuckDBAPI()

In [60]:
# Step 10: Define Blocking Rules

#You will see blocking rules at 3 different sites:
#1. To assess number of matches a combination of columns will produce: this will help to select your model training blocks (Step 10)
#2. Pre-training: Where you ask Splink to use random samples to link. This helps us to learn pre-training estimates (lambda and u-values) (Step 14 and 15)
#3. Session training aka EM-training: Here you train your model and Splink learns about the match weight and probability for all the rows in your data (Step 16)

# Here the we use blocking rules to assess the number of matches a combination of columns will produce
#which is used in the next step of generating cumulative comparisons chart

blocking_rules = [
    block_on('lastname', 'age'),
    block_on('dob', 'age', 'sex'),
    block_on('dob', 'county'),
    block_on('firstname', 'county'),
    block_on('firstname', 'lastname')
]


# Cumulative comparisons chart -- used to assess your blocking rules


cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables = [df_crash, df_hosp],
    blocking_rules = blocking_rules,
    db_api = db_api,
    link_type = 'link_only',
    unique_id_column_name = 'unique_id',
    source_dataset_column_name = 'source').save("cumulative_comparisons.html")

In [62]:
# Step 11: Define custom comparisons for admit date and admit hour to factor in the delay between a crash and hospital admission

#You don't need to remember the detailed code
#The code factors in the following:
#1. if admit date/hour is null in both datasets
#2. if admit date/hour match exactly
#3. if admit date/hour have a difference of 2 days and 48 hours resp between crash and hosp admission

from splink.comparison_library import CustomComparison

admit_date_comp = CustomComparison(
    output_column_name='admit_date_comp',
    comparison_levels=[
        {
            'sql_condition': 'admit_date_r IS NULL OR admit_date_l IS NULL',
            'label_for_charts': 'null',
            'is_null_level': True
        },
        {
            'sql_condition': (
                'admit_date_l IS NOT NULL AND admit_date_r IS NOT NULL AND '
                'CAST(admit_date_l AS DATE) = CAST(admit_date_r AS DATE)'
            ),
            'label_for_charts': 'Exact match on admit_date',
        },
        {
            'sql_condition': (
                "ABS(DATE_DIFF('day', CAST(admit_date_r AS DATE), CAST(admit_date_l AS DATE))) <= 2"
            ),
            'label_for_charts': 'admit_date within 2 days',
        },
        {
            'sql_condition': 'TRUE',
            'label_for_charts': 'all other comparisons'
        }
    ]
)

# Elapsed time between crash and admit: uses date + hour (same 48h rule as before; first match wins).
admit_hr_comp = CustomComparison(
    output_column_name='admit_hr_comp',
    comparison_levels=[
        {
            'sql_condition': (
                'admit_date_r IS NULL OR admit_date_l IS NULL OR '
                'admit_hr_r IS NULL OR admit_hr_l IS NULL'
            ),
            'label_for_charts': 'null',
            'is_null_level': True
        },
        {
            'sql_condition': (
                'admit_date_l IS NOT NULL AND admit_date_r IS NOT NULL AND '
                'admit_hr_l IS NOT NULL AND admit_hr_r IS NOT NULL AND '
                'CAST(admit_date_l AS DATE) = CAST(admit_date_r AS DATE) AND '
                'CAST(admit_hr_l AS DOUBLE) = CAST(admit_hr_r AS DOUBLE)'
            ),
            # Same idea as ExactMatch, but needs BOTH date and hour (hour alone is misleading across days).
            'label_for_charts': 'Exact match (date and hour)',
        },
        {
            'sql_condition': """
                ABS(
                    ((CAST(admit_date_r AS DATE) - CAST(admit_date_l AS DATE)) * 24)
                    + (CAST(admit_hr_r AS FLOAT) - CAST(admit_hr_l AS FLOAT))
                ) <= 48
            """,
            'label_for_charts': 'Within 48 hours (combined date and hour)',
        },
        {
            'sql_condition': 'TRUE',
            'label_for_charts': 'all other comparisons'
        }
    ]
)

In [64]:
# Step 12: Choose comparison methods you wish to use (e.g. ExactMatch)

comparisons = [
        cl.ExactMatch('firstname'), # Can use Jaro winkler or Levenstein
        cl.ExactMatch('lastname'),
        cl.ExactMatch('sex'),
        cl.ExactMatch('dob'),
        cl.ExactMatch('age'),
        cl.ExactMatch('county'),


        admit_date_comp, #Using the custom comparison we created in Step 10
        admit_hr_comp,
    ]

   #the comparison we choose depends on the richness and knowledge of the datasets
   #Example: Customcomparison for admit date and hour is essential to factor in the time required to take a patient to the hosp
   #If you have full_names : can use Jaro winkler or Levenstein
   #But NYS only has first 2 letters of name, which is not very rich, hence exactmatch is required.


In [23]:
#### Put together settings and create Linker object ----------------------------

# Step 13: Define full model settings
model_settings = SettingsCreator(
    unique_id_column_name = 'unique_id',
    link_type='link_only',
    blocking_rules_to_generate_predictions=blocking_rules,
    comparisons=comparisons,
    retain_intermediate_calculation_columns=True,
)

linker = Linker(
    [df_crash, df_hosp],
    model_settings,
    db_api = DuckDBAPI()
)

#### Parameter Estimation ------------------------------------------------------

# Step 14: Pre-training — estimate lambda (probability two random records match)

deterministic_rules = [
    block_on('firstname', 'county'),
    block_on('firstname', 'lastname'),
    block_on('lastname', 'age'),
]

linker.training.estimate_probability_two_random_records_match(
    deterministic_rules, recall=0.8
)

# Step 15: Pre-training — estimate u probabilities using random sampling
# Estimate the prior using a direct estimation technique
# It's recommended to set max pairs high (e.g. 1e7 or more)
# Estimate U probabilities with random sampling

linker.training.estimate_u_using_random_sampling(max_pairs=1e7)


#Till this point we were setting rules/comparisons, assesing blocking rules,
#And asking Splink to estimate pre-training values (lambda and u-values)
#No Linkage has happened yet

INFO:splink.internals.linker_components.training:Probability two random records match is estimated to be  0.00168.
This means that amongst all possible pairwise record comparisons, one in 593.91 are expected to match.  With 1,000,000 total possible comparisons, we expect a total of around 1,683.75 matching pairs
INFO:splink.internals.estimate_u:----- Estimating u probabilities using random sampling -----
INFO:splink.internals.estimate_u:
Estimated u probabilities using random sampling
INFO:splink.internals.settings:
Your model is not yet fully trained. Missing estimates for:
    - firstname (no m values are trained).
    - lastname (no m values are trained).
    - sex (no m values are trained).
    - dob (no m values are trained).
    - age (no m values are trained).
    - county (no m values are trained).
    - admit_date_comp (no m values are trained).
    - admit_hr_comp (no m values are trained).


In [66]:
# Step 16: EM Training — estimate m probabilities

# Estimate Match weight and probabilities with EM algorithm -- requires multiple sessions
# Each session blocks on different variables so all comparisons get trained

#Training sessions (Splink learns values in our data - no linkage)


session_1 = linker.training.estimate_parameters_using_expectation_maximisation(

block_on('lastname', 'age'), estimate_without_term_frequencies=True

)

#In this training session, we blocked on lastname and age,
#splink used these 2 columns to match crash and hosp datasets, using the comparison rules we set
#After matching, it compared rest of the columns (we did not block) and alloted a match weight
#Thus, a match weight was not calculated for lastname and age
#But we need lastname and age to also have its own values, so in the next session we use different columns
#We chose this combinatination based on the cumulative comparisons chart we created earlier


INFO:splink.internals.em_training_session:
----- Starting EM training session -----

INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
(l."lastname" = r."lastname") AND (l."age" = r."age")

Parameter estimates will be made for the following comparison(s):
    - firstname
    - sex
    - dob
    - admit_date_comp
    - admit_hr_comp

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - lastname
    - age
INFO:splink.internals.expectation_maximisation:
INFO:splink.internals.expectation_maximisation:Iteration 1: Largest change in params was -0.38 in probability_two_random_records_match
INFO:splink.internals.expectation_maximisation:Iteration 2: Largest change in params was -0.0495 in the m_probability of admit_date_comp, level `all other comparisons`
INFO:splink.internals.expectation_maximisation:Iteration 3: Largest change in params was -0.00994 in the m_probability of dob,

In [67]:
session_2 = linker.training.estimate_parameters_using_expectation_maximisation(

block_on('firstname', 'county'), estimate_without_term_frequencies=True

)

#Model must have converged after 4-5 iterations which means
#Our m and u probabilities have stopped changing significantly
#And all the columns have atleast have values that can be used to calculate the
#match weight and probability

INFO:splink.internals.em_training_session:
----- Starting EM training session -----

INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
(l."firstname" = r."firstname") AND (l."county" = r."county")

Parameter estimates will be made for the following comparison(s):
    - lastname
    - sex
    - dob
    - age
    - admit_date_comp
    - admit_hr_comp

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - firstname
INFO:splink.internals.expectation_maximisation:
INFO:splink.internals.expectation_maximisation:Iteration 1: Largest change in params was -0.267 in probability_two_random_records_match
INFO:splink.internals.expectation_maximisation:Iteration 2: Largest change in params was 0.00475 in the m_probability of dob, level `Exact match on dob`
INFO:splink.internals.expectation_maximisation:Iteration 3: Largest change in params was -0.00122 in the m_probability of dob, level

In [68]:
session_3 = linker.training.estimate_parameters_using_expectation_maximisation(

block_on('firstname', 'lastname'), estimate_without_term_frequencies=True

)

#Extra session incase your model doesn't converge

INFO:splink.internals.em_training_session:
----- Starting EM training session -----

INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
(l."firstname" = r."firstname") AND (l."lastname" = r."lastname")

Parameter estimates will be made for the following comparison(s):
    - sex
    - dob
    - age
    - admit_date_comp
    - admit_hr_comp

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - firstname
    - lastname
INFO:splink.internals.expectation_maximisation:
INFO:splink.internals.expectation_maximisation:Iteration 1: Largest change in params was -0.394 in probability_two_random_records_match
INFO:splink.internals.expectation_maximisation:Iteration 2: Largest change in params was -0.0301 in the m_probability of admit_date_comp, level `all other comparisons`
INFO:splink.internals.expectation_maximisation:Iteration 3: Largest change in params was -0.00589 in the m_probab

In [29]:
#NEXT STEPS ARE TO ASSESS THE LINKAGE QUALITY

# Step 16: Visualize model parameters

linker.evaluation.unlinkables_chart()

alt.LayerChart(...)

In [30]:
linker.visualisations.match_weights_chart()


alt.VConcatChart(...)

In [31]:
linker.visualisations.m_u_parameters_chart()

alt.HConcatChart(...)

In [32]:
##Step where actual linkage happens

#Step 17: Predict — run linkage and get match probabilities (all blocked pairs)


df_predictions = linker.inference.predict()
df_predictions.as_pandas_dataframe(limit=5)

INFO:splink.internals.linker_components.inference:Blocking time: 0.13 seconds
INFO:splink.internals.linker_components.inference:Predict time: 0.25 seconds


,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,unique_id_r,firstname_l,firstname_r,gamma_firstname,bf_firstname,...,admit_date_r,gamma_admit_date_comp,bf_admit_date_comp,admit_hr_l,admit_hr_r,gamma_admit_hr_comp,bf_admit_hr_comp,county_l,county_r,match_key
0,-16.294936,0.000012,__splink__input_table_0,__splink__input_table_1,crash_742,hosp_1,Kenneth,Kenneth,1,55.483484,...,2024-02-24,0,0.261241,10,0,0,0.355530,westchester,westchester,3
1,-16.294936,0.000012,__splink__input_table_0,__splink__input_table_1,crash_985,hosp_2,Edward,Edward,1,55.483484,...,2024-07-14,0,0.261241,13,5,0,0.355530,queens,queens,3
2,-11.839185,0.000273,__splink__input_table_0,__splink__input_table_1,crash_516,hosp_3,Sandra,Sandra,1,55.483484,...,2024-06-16,0,0.261241,17,23,0,0.355530,westchester,kings,4
3,20.135736,0.999999,__splink__input_table_0,__splink__input_table_1,crash_4,hosp_4,Timothy,Timotiy,0,0.113311,...,2024-11-02,1,45.729334,2,14,1,55.767743,new york,new york,1
4,-16.294936,0.000012,__splink__input_table_0,__splink__input_table_1,crash_134,hosp_5,Steven,Steven,1,55.483484,...,2024-09-29,0,0.261241,2,21,0,0.355530,kings,kings,3


In [39]:
# Step 18: STEPS FOR EXPORTING:

#COLUMNS WE WANT TO EXPORT

cols_keep = [
    'unique_id_l', 'unique_id_r',
    'match_probability', 'match_weight', 'match_key',
    'firstname_l', 'firstname_r','lastname_l',  'lastname_r',
    'dob_l', 'dob_r', 'sex_l',  'sex_r',  'age_l', 'age_r',

    'county_l', 'county_r'
    'role_l', 'role_r',
    'admit_date_l', 'admit_date_r',
    'admit_hr_l', 'admit_hr_r',
]

In [40]:
#EXPORTS YOUR ENTIRE LINKAGE (irrespective of match probability)

df_predictions_pd = df_predictions.as_pandas_dataframe()

df_predictions_export = df_predictions_pd [[c for c in cols_keep if c in df_filtered.columns]]

df_predictions_export.to_csv('synthetic_linkage_results.csv', index=False)
print(f"Matched records saved: {len(df_predictions_export)} rows")

Matched records saved: 1388 rows


In [41]:
#Can filter based on match probability and export as .csv

df_filtered = df_predictions_pd[df_predictions_pd['match_probability'] >= 0.8]

df_filtered = df_filtered[[c for c in cols_keep if c in df_filtered.columns]]

df_filtered.to_csv('synthetic_linkage_results_filtered.csv', index=False)

print(f"Matched records saved: {len(df_filtered)} rows")

Matched records saved: 335 rows
